In [5]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf torch accelerate bitsandbytes

In [38]:
import os
import torch
import warnings
import re
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from pypdf import PdfReader
from typing import List, Dict, Tuple

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# --- 1. Upload Assignment Files (Reverted to ensure upload prompt) ---
print("Please upload 'apple_doc.pdf' and 'tesla_doc.pdf' now.")
try:
    # Use files.upload() to prompt the user
    uploaded = files.upload()
    print("Files uploaded successfully!")
except Exception as e:
    print(f"Error during file upload. Please ensure files are selected. Error: {e}")

# Define file paths
APPLE_DOC = "apple_doc.pdf"
TESLA_DOC = "tesla_doc.pdf"

# --- 2. Custom Data Structure for Chunks ---
Chunk = Tuple[str, str] # (text_content, metadata_source_string)

# --- 3. Custom Prompt Template (Aggressively Revised for Final Output) ---
RAG_PROMPT_TEMPLATE = """
You are a RAG system and your ONLY job is to extract the answer and citation from the CONTEXT.

## STRICT OUTPUT RULES:
1. **Answer ONLY from the provided CONTEXT.** Do not use external knowledge.
2. **Output MUST be a single, concise string: [ANSWER] [CITATION].** No pre-amble, no explanation, no extra text.
3. **Citation Format:** [Document Name, Item/Note/Page Number]. E.g., 'The revenue was $100 billion [Apple 10-K, Item 7, Page 21].'
4. **Out-of-Scope/Future Dates:** If the context does not contain the answer, or if the question is opinion-based or asks about a date *after* the document's filing date, the **ENTIRE RESPONSE MUST BE**: "Not answerable"

CONTEXT:
{context}

QUESTION: {question}

FINAL ANSWER:
"""

Please upload 'apple_doc.pdf' and 'tesla_doc.pdf' now.


Saving apple_doc.pdf to apple_doc (2).pdf
Saving tesla_doc.pdf to tesla_doc (2).pdf
Files uploaded successfully!


In [14]:
# --- 1. Document Splitter Function (FIXED) ---
def simple_recursive_splitter(text: str, chunk_size: int, chunk_overlap: int, source: str) -> List[Chunk]:
    """Splits text into chunks with overlap, returning (content, source) tuples."""

    # Custom recursive splitting logic based on newline/space separators
    def split_by_separators(text, separators):
        parts = [text]
        for sep in separators:
            # Skip empty separators if they appear due to text cleaning issues
            if not sep:
                continue

            new_parts = []
            for part in parts:
                if part and isinstance(part, str):
                    new_parts.extend([p for p in part.split(sep) if p])
            parts = new_parts
        return parts

    separators = ["\n\n", "\n", " "] # Removed the empty string separator "" which was the source of the error

    # We use a space as the final implicit separator to tokenize words
    words = split_by_separators(text, separators)

    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 <= chunk_size:
            current_chunk += (" " if current_chunk else "") + word
        else:
            if current_chunk:
                chunks.append((current_chunk, source))
                # Create overlap by moving back a set amount of characters/words
                overlap_text = current_chunk[-chunk_overlap:] if chunk_overlap > 0 else ""
                current_chunk = overlap_text + (" " if overlap_text else "") + word
            else:
                # Handle case where a single word is larger than chunk_size (rare)
                chunks.append((word, source))
                current_chunk = ""

    if current_chunk:
        chunks.append((current_chunk, source))

    return chunks

# --- 2. Load Documents and Parse Text ---
def load_and_chunk_pdf(file_path: str, doc_name: str) -> List[Chunk]:
    """Loads a PDF, extracts text page by page, and chunks it."""
    reader = PdfReader(file_path)
    all_chunks = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()

        # Skip pages if text extraction fails or returns non-string content
        if not text or not isinstance(text, str):
            continue

        page_source = f"{doc_name}, Page {i + 1}"

        # Split text into chunks, preserving metadata
        chunks = simple_recursive_splitter(text, chunk_size=1000, chunk_overlap=150, source=page_source)
        all_chunks.extend(chunks)

    return all_chunks

# Load and split documents
print("Loading and chunking Apple 10-K...")
apple_chunks = load_and_chunk_pdf(APPLE_DOC, "Apple 10-K")
print("Loading and chunking Tesla 10-K...")
tesla_chunks = load_and_chunk_pdf(TESLA_DOC, "Tesla 10-K")

all_chunks: List[Chunk] = apple_chunks + tesla_chunks
print(f"Total chunks created: {len(all_chunks)}")

# --- 3. Embedding Model and FAISS Index (Core Libraries) ---
embed_model_name = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(embed_model_name, device='cuda' if torch.cuda.is_available() else 'cpu')

print("Generating embeddings and creating FAISS Vector Store...")
# Separate text content from source strings for embedding
chunk_contents = [content for content, source in all_chunks]
chunk_sources = [source for content, source in all_chunks]

# Generate embeddings
embeddings = embedder.encode(chunk_contents, convert_to_tensor=True, show_progress_bar=True).cpu().numpy()
d = embeddings.shape[1]  # Dimension of the embeddings

# Create FAISS Index
index = faiss.IndexFlatL2(d)
index.add(embeddings)
print("Vector Store created and indexed successfully.")

Loading and chunking Apple 10-K...
Loading and chunking Tesla 10-K...
Total chunks created: 1111
Generating embeddings and creating FAISS Vector Store...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Vector Store created and indexed successfully.


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from sentence_transformers import CrossEncoder

# --- 1. Open-Source LLM Setup (Mistral 7B) ---
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define 4-bit quantization configuration for maximum memory savings
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading LLM with 4-bit quantization...")
# FIX APPLIED: Removed the invalid 'llm_int8_enable_fp32_cpu_offload' argument.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    trust_remote_code=True,
    device_map="auto",
    do_sample=False,
)
print("LLM (Mistral 7B) pipeline loaded.")

# --- 2. Re-Ranker Model Setup (BGE Re-ranker) ---
reranker_model = CrossEncoder(
    'BAAI/bge-reranker-base',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
print("Re-Ranker (BGE) model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading LLM with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM (Mistral 7B) pipeline loaded.
Re-Ranker (BGE) model loaded.


In [39]:
def answer_question(query: str) -> Dict[str, str]:
    """
    Answers a question using the RAG pipeline with search, re-ranking, and LLM generation.

    Args:
        query: The question to be answered.

    Returns:
        A dictionary with the final answer and its source.
    """

    # 1. Retrieval (FAISS Search)
    k_initial = 15

    # Embed the query
    query_embedding = embedder.encode(query, convert_to_tensor=True).cpu().numpy()

    # Perform FAISS search (D = distance, I = indices)
    D, I = index.search(query_embedding.reshape(1, -1), k_initial)

    # Gather retrieved documents
    retrieved_results = []
    for idx in I[0]:
        if idx != -1: # Ensure index is valid
            content = chunk_contents[idx]
            source = chunk_sources[idx]
            retrieved_results.append((content, source))

    if not retrieved_results:
        return {"answer": "Not answerable", "source": "$N/A$"}

    # 2. Re-Ranking (Step 2 Requirement)

    # Re-ranker input: list of (query, document) pairs
    reranker_input = [(query, content) for content, source in retrieved_results]
    reranker_scores = reranker_model.predict(reranker_input)

    # Combine content, source, and new scores
    reranked_data = []
    for (content, source), score in zip(retrieved_results, reranker_scores):
        reranked_data.append({'text': content, 'source': source, 'score': score})

    # Sort by the re-ranker score (highest score is best) and select top-5
    reranked_data.sort(key=lambda x: x['score'], reverse=True)
    top_k_final = 5
    top_chunks = reranked_data[:top_k_final]

    # 3. Context Formatting for LLM
    context = ""
    for chunk in top_chunks:
        # Source: [Source String] Text: [Content]
        context += f"Source: {chunk['source']}\n"
        context += f"Text: {chunk['text']}\n\n"

    # 4. LLM Integration and Generation
    full_prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=query)

    try:
        # Use the HuggingFace pipeline for generation
        response = llm_pipeline(full_prompt, max_new_tokens=256, do_sample=False, num_return_sequences=1)[0]['generated_text']

        # Clean up the response to isolate the LLM's final answer
        answer_text = response.split("FINAL ANSWER:")[-1].strip()

        # Heuristically check the LLM's output for the required structure
        if answer_text.startswith("Not answerable"):
            final_answer = "Not answerable"
            final_source = "$N/A$"
        else:
            final_answer = answer_text
            # Use the actual answer text to set the simplified source (less error-prone)
            if "Apple" in final_answer:
                final_source = "Apple 10-K"
            elif "Tesla" in final_answer:
                final_source = "Tesla 10-K"
            else:
                final_source = "Source included in answer text"

        return {"answer": final_answer, "source": final_source}

    except Exception as e:
        print(f"An error occurred during LLM generation: {e}")
        return {"answer": "An internal error occurred.", "source": "$N/A$"}

In [40]:
import json
import re

# Define the official list of questions
official_questions = [
    {"question_id": 1, "question": "What was Apple's total revenue for the fiscal year ended September 28, 2024?"},
    {"question_id": 2, "question": "How many shares of common stock were issued and outstanding as of October 18, 2024?"},
    {"question_id": 3, "question": "What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?"},
    {"question_id": 4, "question": "On what date was Apple's 10-K report for 2024 signed and filed with the SEC?"},
    {"question_id": 5, "question": "Does Apple have any unresolved staff comments from the SEC as of this filing? How do you know?"},
    {"question_id": 6, "question": "What was Tesla's total revenue for the year ended December 31, 2023?"},
    {"question_id": 7, "question": "What percentage of Tesla's total revenue in 2023 came from Automotive Sales (excluding Leasing)?"},
    {"question_id": 8, "question": "What is the primary reason Tesla states for being highly dependent on Elon Musk?"},
    {"question_id": 9, "question": "What types of vehicles does Tesla currently produce and deliver?"},
    {"question_id": 10, "question": "What is the purpose of Tesla's ’lease pass-through fund arrangements’?"},
    {"question_id": 11, "question": "What is Tesla's stock price forecast for 2025?"},
    {"question_id": 12, "question": "Who is the CFO of Apple as of 2025?"},
    {"question_id": 13, "question": "What color is Tesla's headquarters painted?"}
]

# Function to process the answer and extract citations (IMPROVED)
def format_rag_output(question_id: int, question_text: str, rag_result: Dict[str, str]) -> Dict:
    """Formats the raw RAG output into the required JSON structure."""

    final_answer = rag_result['answer']

    if final_answer.startswith("Not answerable"):
        # Case 1: Unanswerable/Out of Scope
        return {
            "question_id": question_id,
            "answer": "This question cannot be answered based on the provided documents.",
            "sources": []
        }
    else:
        # Case 2: Answerable - Extract sources

        # Regex to find citations (looking for anything inside square brackets)
        citations = re.findall(r'\[([^\]]+)\]', final_answer)

        # Clean the answer by removing the last bracketed citation for a clean answer text
        # But we keep all citations in the sources list

        clean_answer = final_answer

        # Remove any citation string from the end of the answer string for cleaner presentation
        for citation in reversed(citations):
            full_cite_string = f"[{citation}]"
            if clean_answer.endswith(full_cite_string):
                clean_answer = clean_answer[:-len(full_cite_string)].strip()
                break # Only remove the last one to leave the answer intact

        return {
            "question_id": question_id,
            "answer": final_answer, # Output the full LLM string including the citation for the final JSON
            "sources": citations # Output the cleaned source list
        }

# --- Execution Loop ---
print("--- Running Assignment Test Cases and Generating JSON Output ---")
json_results = []

for item in official_questions:
    q_id = item['question_id']
    q_text = item['question']

    print(f"\n--- Processing Question {q_id}: {q_text} ---")

    # Call the RAG function (defined in Cell 5)
    result = answer_question(q_text)

    # Format the result into the required JSON structure
    formatted_output = format_rag_output(q_id, q_text, result)
    json_results.append(formatted_output)

    # Print the raw JSON for immediate visibility
    print(json.dumps(formatted_output, indent=4))
    print("-" * (len(q_text) + 20))

# --- Final Output ---
print("\n\n--- Final Consolidated JSON Output ---")
# Print the final, complete JSON list
print(json.dumps(json_results, indent=4))

--- Running Assignment Test Cases and Generating JSON Output ---

--- Processing Question 1: What was Apple's total revenue for the fiscal year ended September 28, 2024? ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{
    "question_id": 1,
    "answer": "391,035 [Apple 10-K, Page 38, Text]\n\nCITATION:\n[Apple 10-K, Page 38, Text]",
    "sources": [
        "Apple 10-K, Page 38, Text",
        "Apple 10-K, Page 38, Text"
    ]
}
------------------------------------------------------------------------------------------------

--- Processing Question 2: How many shares of common stock were issued and outstanding as of October 18, 2024? ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{
    "question_id": 2,
    "answer": "15,115,823,000 shares [Apple 10-K, Page 2]\n\nCITATION:\n[Apple 10-K, Page 2]",
    "sources": [
        "Apple 10-K, Page 2",
        "Apple 10-K, Page 2"
    ]
}
-------------------------------------------------------------------------------------------------------

--- Processing Question 3: What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024? ---


...

{
    "question_id": 11,
    "answer": "This question cannot be answered based on the provided documents.",
    "sources": []
}
------------------------------------------------------------------

--- Processing Question 12: Who is the CFO of Apple as of 2025? ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{
    "question_id": 12,
    "answer": "Luca Maestri [Apple 10-K, Page 118, Signature of Luca Maestri]\n\nCITATION:\n[Apple 10-K, Page 118, Signature of Luca Maestri]",
    "sources": [
        "Apple 10-K, Page 118, Signature of Luca Maestri",
        "Apple 10-K, Page 118, Signature of Luca Maestri"
    ]
}
-------------------------------------------------------

--- Processing Question 13: What color is Tesla's headquarters painted? ---


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{
    "question_id": 13,
    "answer": "This question cannot be answered based on the provided documents.",
    "sources": []
}
---------------------------------------------------------------


--- Final Consolidated JSON Output ---
[
    {
        "question_id": 1,
        "answer": "391,035 [Apple 10-K, Page 38, Text]\n\nCITATION:\n[Apple 10-K, Page 38, Text]",
        "sources": [
            "Apple 10-K, Page 38, Text",
            "Apple 10-K, Page 38, Text"
        ]
    },
    {
        "question_id": 2,
        "answer": "15,115,823,000 shares [Apple 10-K, Page 2]\n\nCITATION:\n[Apple 10-K, Page 2]",
        "sources": [
            "Apple 10-K, Page 2",
            "Apple 10-K, Page 2"
        ]
    },
    {
        "question_id": 3,
        "answer": "The total amount of term debt reported by Apple as of September 28, 2024 is $100,544 million (Apple 10-K, Page 34) + $97,341 million (Apple 10-K, Page 46) = $197,885 million.\n\nCITATION:\nApple Inc. | 2024 Form 10-K | Page 34, 